In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

# Tools must inherit BaseTool

@tool # -> wraps the function inside StructuredTool
def add(a, b):
    """ Add two numbers passed as arguments """
    return a + b

@tool
def subtract(a, b):
    """ Subtract two numbers passed as arguments """
    return a - b
@tool
def multiply(a, b):
    """ Multiply two numbers passed as arguments """
    return a * b

tavily_search = TavilySearch(max_results=5)
tools = [add, subtract, multiply, tavily_search]


In [ ]:
from langchain_openai import ChatOpenAI


model = ChatOpenAI(model="gpt-5-nano")
llm_with_tools = model.bind_tools(tools)

In [ ]:
response = llm_with_tools.invoke("add 2 and 4")
response

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.messages import SystemMessage

template = ChatPromptTemplate(messages=[
    SystemMessage(content="Don't hallucinate, just use provided tools"),
    HumanMessagePromptTemplate.from_template("{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])


agent = create_tool_calling_agent(llm=model, tools=tools, prompt=template)

executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
executor.invoke({'input': "add 1 and 2 then multiply with 5"})